In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")
print("GROQ key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("COHERE key loaded:", bool(os.getenv("COHERE_API_KEY")))
print("PINECONE key loaded:", bool(os.getenv("PINECONE_API_KEY")))

/var/folders/xj/86qx78n90950d4pfdy9105nm0000gn/T/ipykernel_65837/4272355400.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


All imports successful!
GROQ key loaded: True
COHERE key loaded: True
PINECONE key loaded: True


In [2]:
# Drop any PDF into the /data folder and point to it here
loader = PyMuPDFLoader("../data/sample.pdf")
documents = loader.load()

print(f"Loaded {len(documents)} pages")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,       # max characters per chunk
    chunk_overlap=64,     # overlap so context isn't lost at boundaries
    separators=["\n\n", "\n", ".", " "]  # try to split at natural breaks
)

chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")
print(f"\nSample chunk:\n{chunks[0].page_content[:300]}")

Loaded 34 pages
Split into 134 chunks

Sample chunk:
Multilingual Sentiment Analysis 
 
Project report submitted to the Indian Institute of Information Technology, Nagpur, in 
partial fulfilment of the requirements for the Award of the Degree of 
 
BACHELOR OF TECHNOLOGY 
In 
 
COMPUTER SCIENCE AND ENGINEERING 
 
by 
Anuj Singh (BT22CSE135)  
        


In [3]:
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=os.getenv("COHERE_API_KEY")
)

# Clear old vectors from Pinecone before inserting new ones
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(os.getenv("PINECONE_INDEX_NAME"))
index.delete(delete_all=True)  # wipe everything
print("Cleared old vectors from Pinecone")

# Now store the new PDF chunks
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=os.getenv("PINECONE_INDEX_NAME")
)

print(f"Stored {len(chunks)} new chunks in Pinecone!")

Cleared old vectors from Pinecone
Stored 134 new chunks in Pinecone!


In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

prompt = PromptTemplate.from_template("""<instructions>
You are a document assistant. You MUST follow these rules strictly:
1. Answer ONLY using the context provided below.
2. If the context does not contain the answer, respond with exactly: "The document does not contain information about this."
3. Do NOT use any outside knowledge. Do NOT guess.
4. Quote directly from the context where possible.
5. Cite the source and page number after every claim.
</instructions>

<context>
{context}
</context>

<question>{question}</question>

<answer>""")

def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata.get('source', 'unknown')} p.{doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Chain ready!")

Chain ready!


In [5]:
question = "What is this document about?"  # change to your actual question

# Step 1 — see exactly what chunks the retriever is finding
retrieved_docs = retriever.invoke(question)
print(f"=== {len(retrieved_docs)} chunks retrieved ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}")
    print(f"Content preview: {doc.page_content[:300]}")
    print()

# Step 2 — now get the actual answer
print("\n=== LLM ANSWER ===")
answer = chain.invoke(question)
print(answer)

=== 5 chunks retrieved ===

--- Chunk 1 ---
Source: ../data/sample.pdf | Page: 33.0
Content preview: (ICACCI), 2015, pp. 1468–1473. 
[18]​
G. Molina et al., “Overview for the Second Shared Task on Language Identification in 
Code-Switched Data,” in Proc. 2nd Workshop on Computational Approaches to Code Switching, 2016, pp. 
40–49. 
[19]​
K. Goswami, P. Rani, B. R. Chakravarthi, T. Fransen, and J. P

--- Chunk 2 ---
Source: ../data/sample.pdf | Page: 32.0
Content preview: [5]​[Author Unknown], “Title unknown,” arXiv, 2021. Available: https://arxiv.org/abs/2103.XXXXX 
 
[6]​L. Advani, C. Lu, and S. Maharjan, “C1 at SemEval-2020 Task 9: SentiMix: Sentiment Analysis for 
Code-Mixed Social Media Text Using Feature Engineering,” in Proc. 14th Workshop on Semantic Evaluati

--- Chunk 3 ---
Source: ../data/sample.pdf | Page: 32.0
Content preview: Evaluation, 2020, pp. 928–933. 
[8]​
J. Kong, J. Wang, and X. Zhang, “HPCC-YNU at SemEval-2020 Task 9: A Bilingual Vector 
Gating Mechanism for Senti

In [12]:
import os
print("Jupyter is running from:", os.getcwd())


Jupyter is running from: /Users/anuj/Desktop/PJ/rag-assistant/notebooks


In [10]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current folder:
/Users/anuj/Desktop/PJ/rag-assistant/notebooks

Files:
['Untitled.ipynb', '.ipynb_checkpoints']


In [17]:
import sys
import os

# Point to project root regardless of where Jupyter is running from
project_root = "/Users/anuj/Desktop/PJ/rag-assistant"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Python path includes:", project_root)
print("Backend folder exists:", os.path.exists(os.path.join(project_root, "backend")))
print("ingest.py exists:", os.path.exists(os.path.join(project_root, "backend", "ingest.py")))

from backend.ingest import ingest_file
print("Import successful!")


Python path includes: /Users/anuj/Desktop/PJ/rag-assistant
Backend folder exists: True
ingest.py exists: True
Import successful!


In [19]:
result = ingest_file("../data/sample1.docx")
print(result)

Indexing 'sample1.docx'...
  Loaded 1 pages/sections
  Split into 151 chunks
  Stored 151 chunks in Pinecone
{'doc_id': '24fad33d0d83ee3c54c6c4ee54288697', 'filename': 'sample1.docx', 'chunks': 151, 'skipped': False}
